# Project 5 — Module 6: Supervised Machine Learning
## Lesson 1: Fundamentals of Machine Learning

| | |
|---|---|
| **Author** | Jose Marcel Lopez Pino |
| **Framework** | CRISP-DM + LEAN + DMAIC (Define) |
| **Phase** | 1 — Business Understanding |
| **Module** | 6 — Supervised Machine Learning (Alkemy Bootcamp) |
| **Dataset** | PequeShop — customers_final.csv + transactions_final.csv |
| **Date** | 2026-03 |

---

> **Executive Summary:**  
> This notebook defines the business problem for PequeShop's supervised ML
> pipeline using a Problem Statement Canvas — the standard problem framing
> tool for Data Science (CRISP-DM + LEAN + DMAIC Define + Decision
> Science). The target variable is `avg_ticket` (average purchase amount per
> customer). Statistical evidence from project-4b directly informs feature
> selection, eliminating waste before modeling begins.
> The M6 consigna requests features such as age and site behavior;
> PequeShop equivalent proxies are used and documented in Decisions Log #7.


## Table of Contents

1. [CRISP-DM Phase 1 — Business Understanding](#1-crisp-dm-phase-1--business-understanding)
2. [Problem Statement Canvas](#2-problem-statement-canvas)
3. [Classification vs Regression — Conceptual Differentiation](#3-classification-vs-regression--conceptual-differentiation)
4. [Evidence from Project-4b — Statistical Foundation](#4-evidence-from-project-4b--statistical-foundation)
5. [ML Pipeline — End-to-End Map](#5-ml-pipeline--end-to-end-map)
6. [Target Variable & Feature Candidates](#6-target-variable--feature-candidates)
7. [LEAN Filter — Waste Elimination Review](#7-lean-filter--waste-elimination-review)
8. [Decisions Log — Lesson 1](#8-decisions-log--lesson-1)
9. [Next Steps — Lesson 2 Preview](#9-next-steps--lesson-2-preview)


---
## 1. CRISP-DM Phase 1 — Business Understanding

### 1.1 Context

**PequeShop** is a Chilean children's e-commerce business selling across
MercadoLibre and Shopify. This project is the fourth cycle of a continuous
CRISP-DM analysis over the same dataset:

| Project | Scope | Output |
|---------|-------|--------|
| `project-2-pequeshop-analytics` | ETL + Data Understanding | `customers_final.csv`, `transactions_final.csv` |
| `project-3-eda-pequeshop` | Exploratory Data Analysis | Diagnostic insights, distributions, correlations |
| `project-4b-pequeshop-statistical-inference` | Statistical Inference | H1–H4 hypothesis tests, effect sizes, CI |
| **`project-5-ecommerce-spend-prediction`** | **Supervised ML** | **Predictive model for avg_ticket** |

> **Lean principle — no re-work:** Data preparation, EDA, and statistical
> inference are complete. This project builds directly on those outputs.
> No data cleaning, no EDA repetition, no re-testing of H1–H4.

### 1.2 Problem Framing Approach

This project uses a **Problem Statement Canvas** — the standard tool for
business analytics problem framing. It is grounded in three frameworks:

| Framework | Contribution |
|-----------|-------------|
| **DMAIC Define** | Structured problem and impact definition |
| **CRISP-DM Phase 1** | Business understanding before any data work |
| **Decision Science** | Connects model output to actionable business decisions |

> **Why not PICO?** PICO (Patient, Intervention, Comparison, Outcome) is
> designed for clinical research and Evidence-Based Medicine. It is not
> appropriate for business analytics or ML projects.


---
## 2. Problem Statement Canvas

### 2.1 The Canvas

| Element | Content |
|---------|----------|
| **1. Business Problem** | PequeShop cannot personalize marketing offers — all customers receive the same promotions despite significantly different spending behavior across platforms and segments |
| **2. Business Impact** | avg_ticket deviates significantly from the $25,000 CLP market benchmark (H1: t=7.80, p<0.001). Churn exceeds 30% industry threshold (H3: z=5.18, p<0.001). MercadoLibre and Shopify customers spend differently (H2: t=2.27, p=0.024). Estimated impact: ~45 customers lost per cycle without targeted retention |
| **3. Decision to Support** | Two business decisions: (1) Marketing needs to decide which customers should receive high-value retention offers vs standard promotions — based on predicted spend, not gut feeling. (2) Ads/retargeting budget should be concentrated on customers with predicted avg_ticket above a profitability threshold — improving promotion ROI by avoiding spend on low-value customers |
| **4. Analytical Question** | Can we predict `avg_ticket` per customer from platform, behavioral, and segment data with sufficient accuracy to justify personalized marketing investment? |
| **5. Success Metrics** | R² > 0.70 on test set · MAPE < 20% · RMSE < 20% of mean avg_ticket · MAE improvement > 20% over mean-predictor baseline |
| **6. Proposed Approach** | Supervised regression using customer and transaction features — Linear Regression (baseline) → Ridge/Lasso (regularization) → GradientBoosting (ensemble). Features pre-selected using project-4b statistical evidence |

### 2.2 Chain of Reasoning

```
Business Problem:  Customers treated equally despite different spending behavior
        ↓
Decision to Improve:  Which customers deserve personalized high-value offers?
        ↓
Metrics:  avg_ticket per customer (CLP)
        ↓
Data:  customers_final.csv + transactions_final.csv (392 customers, 1,192 transactions)
        ↓
Model:  Supervised regression — Linear → Ridge → GradientBoosting
        ↓
Business Impact:  Targeted retention campaigns + optimized ad spend for high-value customers → improved promotion ROI
```

> **Note:** Projects that start at "Model" look junior.
> Projects that start at "Business Problem" look senior.
> This canvas ensures PequeShop's ML pipeline is decision-driven, not model-driven.


---
## 3. Classification vs Regression — Conceptual Differentiation

### 3.1 Supervised Learning Problem Types

| Dimension | Classification | Regression |
|-----------|---------------|------------|
| **Output** | Discrete category (class label) | Continuous numeric value |
| **Question** | "Which group does this belong to?" | "How much / how many?" |
| **PequeShop examples** | Churn: Yes/No · NPS: Promoter/Passive/Detractor | avg_ticket in CLP · total_revenue |
| **Loss function** | Cross-entropy, accuracy | MAE, MSE, RMSE |
| **Evaluation metrics** | Precision, Recall, F1, ROC-AUC | R², MAE, RMSE, MAPE |

### 3.2 Why This is a Regression Problem

| Question | Answer |
|----------|--------|
| Is the output a category? | ❌ No — it is a CLP amount (continuous) |
| Is the output ordered and continuous? | ✅ Yes |
| Does RMSE make sense as loss? | ✅ Yes |
| Could we use classification? | ⚠️ Only by discretizing into tiers — loses precision, not recommended |

> **Conclusion:** Predicting `avg_ticket` is a **supervised regression** problem.
> Classification applies to churn prediction (H3) — a different task outside
> this project's scope.


---
## 4. Evidence from Project-4b — Statistical Foundation

`project-4b-pequeshop-statistical-inference` (Module 5 — Statistical Inference)
validated which variables have a statistically significant relationship with
`avg_ticket` using formal hypothesis testing (H1–H4, α=0.05). This eliminates
guesswork in feature selection.

| Hypothesis | Result | ML Implication |
|-----------|--------|----------------|
| H1 — avg_ticket vs $25k | Rejected (t=7.80, p<0.001) | Target variable has meaningful variation — worth predicting ✅ |
| H2 — MercadoLibre vs Shopify | Rejected (t=2.27, p=0.024) | `platform` is a significant feature ✅ include |
| H3 — Churn rate vs 30% | Rejected (z=5.18, p<0.001) | `retargeting_segment` is a significant feature ✅ include |
| H4 — avg_ticket by NPS | Not rejected (F=0.25, p=0.780) | `nps_category` does NOT predict ticket ❌ low priority |

> **Lean principle:** Statistical evidence eliminates feature selection waste.
> We do not need to test all possible features — we already know which ones matter.


---
## 5. ML Pipeline — End-to-End Map

### 5.1 Prior Work — No Re-Work Principle

Project-5 is the fourth cycle of a continuous CRISP-DM pipeline over
PequeShop. Each prior project delivered specific outputs that feed directly
into this ML project:

| Prior Project | CRISP-DM Work Done | Key Output for Project-5 |
|--------------|-------------------|---------------------------|
| `project-2-pequeshop-analytics` | ETL + data cleaning + feature engineering | `customers_final.csv`, `transactions_final.csv` — ready to use |
| `project-3-eda-pequeshop` | Full EDA — distributions, correlations, outlier analysis | Feature candidates identified |
| `project-4b-pequeshop-statistical-inference` | Hypothesis testing H1–H4 (α=0.05) — **statistical validation of features** | Feature selection confirmed by evidence — see section 4 |

> **Why statistical inference before ML?**  
> `project-4b` formally tested which variables have a statistically
> significant relationship with `avg_ticket`. This is not academic exercise —
> it is evidence-based feature selection. H2 confirmed `platform`, H3
> confirmed `retargeting_segment`, H4 eliminated `nps_category`.
> Project-5 builds its feature matrix on this foundation.

> **Lean rule:** Never repeat work already completed in a previous project
> of the same cycle. Reference it, build on it.

### 5.2 Project-5 Scope — What is New

Only work that is **new and specific to supervised ML** is performed here.
8 lessons from the M6 consigna consolidated into 6 CRISP-DM notebooks:

| Notebook | CRISP-DM Phase | Lessons | Scope |
|----------|---------------|------------|-------|
| 01 | Business Understanding | L1 | Problem Statement Canvas, Classification vs Regression, Pipeline |
| 02 | Data Understanding | L2 | EDA, train/test split, K-Folds cross-validation, overfitting diagnosis |
| 03 | Data Preparation | L3 | Nulls, outliers, encoding, scaling |
| 04 | Modeling | L4 + L5 | Linear, Polynomial, KNN (contrast) |
| 05 | Evaluation | L6 + L7 | MAE, RMSE, MAPE, R², GridSearchCV, Ridge, Lasso |
| 06 | Deployment | L8 | GradientBoosting, final model, business recommendations |

```
01 Business Understanding
    ↓
02 Data Understanding  (EDA + K-Folds setup)
    ↓
03 Data Preparation    (encoding + scaling)
    ↓
04 Modeling            (Linear + Polynomial + KNN)
    ↓
05 Evaluation          (Metrics + GridSearchCV + Ridge + Lasso)
    ↓
06 Deployment          (GradientBoosting + final model + business impact)
```


---
## 6. Target Variable & Feature Candidates

### 6.1 Target Variable

| Variable | Type | Source | Description |
|----------|------|--------|-------------|
| `avg_ticket` | Continuous (CLP) | customers_final.csv | Average purchase amount per customer |

### 6.2 Feature Candidates

| Feature | Type | Evidence | Priority |
|---------|------|----------|----------|
| `primary_platform` | Categorical | H2 rejected ✅ | HIGH |
| `retargeting_segment` | Categorical | H3 rejected ✅ | HIGH |
| `total_transactions` | Numeric | Transaction volume proxy | MEDIUM |
| `total_revenue` | Numeric | Correlated with ticket | MEDIUM |
| `is_bulk_order` | Binary | Higher ticket signal | MEDIUM |
| `season` | Categorical | Seasonal patterns | LOW |
| `nps_category` | Categorical | H4 not rejected ❌ | LOW |

> Final feature selection confirmed in notebook 03 after correlation analysis.


---
## 7. LEAN Filter — Waste Elimination Review

| LEAN Question | Answer | Action |
|---------------|--------|--------|
| Does predicting avg_ticket enable a business decision? | ✅ Yes — personalized offers, channel strategy, retention | Keep |
| Is Problem Statement Canvas better than PICO for this project? | ✅ Yes — PICO is clinical, Canvas is business analytics | Use Canvas |
| Does project-4b evidence reduce feature selection waste? | ✅ Yes — H1-H4 already tested which variables matter | Apply it |
| Should we include NPS given H4 result? | ⚠️ Low priority — no statistical justification | Test last |
| Is the baseline model necessary? | ✅ Yes — without baseline, model performance has no reference | Always include |


---
## 8. Decisions Log — Lesson 1

| # | Decision | Rationale | Alternatives Considered | LEAN Value? |
|---|----------|-----------|------------------------|-------------|
| 1 | Use Problem Statement Canvas instead of PICO | The PICO framework (Patient, Intervention, Comparison, Outcome) is designed for clinical research within evidence-based medicine — used in project-4 (student health). This project addresses a business analytics problem in e-commerce; a Problem Statement Canvas provides a more appropriate structure for defining the business problem and decision context. | Keep PICO | ✅ Correct problem framing tool |
| 2 | Frame as regression, not classification | avg_ticket is continuous — classification requires arbitrary binning and loses precision | Discretize into Low/Medium/High spend tiers | ✅ Statistically correct |
| 3 | Use PequeShop dataset (synthetic) | Continuity with project-4b — same data, progressive CRISP-DM cycle | Use external Kaggle dataset | ✅ Portfolio narrative continuity |
| 4 | Set R² > 0.70 as success threshold | Industry standard for business-grade regression | R² > 0.60 (too lenient) | ✅ Honest and measurable |
| 5 | Exclude NPS as primary feature | H4: F=0.25, p=0.780 — no predictive value for avg_ticket | Include NPS regardless | ✅ Evidence-based feature selection |
| 6 | Consolidate 8 lessons into 6 CRISP-DM notebooks | CRISP-DM phases are the organizing principle — cleaner structure | One notebook per lesson | ✅ Professional structure |


---
## 9. Next Steps — Lesson 2 Preview

In **Lesson 2 — Data Understanding (notebook 02)**, the following tasks will
be completed:

1. Load and explore `customers_final.csv` and `transactions_final.csv`
2. Analyze `avg_ticket` distribution and outliers
3. Explore feature correlations with target variable
4. Train/test split (80/20, random_state=42)
5. Implement K-Folds cross-validation (k=5)
6. Diagnose overfitting/underfitting risks from data characteristics

---

**Next Phase -->** [02 - Data Understanding](./02_data_understanding.ipynb)

> **Project-6 preview (Module 7 — Unsupervised ML):** Once avg_ticket prediction
> is validated in project-5, the next cycle (`project-6-pequeshop-customer-segmentation`)
> will apply clustering and dimensionality reduction to discover hidden customer segments —
> answering *"Who are our customers?"* to enable differentiated retargeting,
> loyalty, and cross-sell campaigns.

*Framework: CRISP-DM + Lean + DMAIC | Module 6 — Supervised Machine Learning*  
*Jose Marcel Lopez Pino — Bootcamp Fundamentos de Ciencia de Datos, SENCE/Alkemy 2025-2026*
